# National Park Visitation in Washington, D.C. 🏛️

**An exploratory analysis of where and when people visit National Park Service sites in the Washington, D.C. area.**

This project uses publicly sourced National Park Service visitation data to examine site popularity, seasonal patterns, and longer-term changes. The goal is to identify patterns that could inform visitor outreach and planning.

*Tools: Python · pandas · Plotly Express*

## 📋 1. Project Overview

### Questions
1. How many NPS sites and recreation visits are represented in the 2025 data?
2. Which sites account for the largest share of visits?
3. When are recreation visits highest during the year?
4. How does 2025 fit into the longer-term visitation trend?

### Data
Three CSV files originally assembled for an NPS analysis LiveLab, sourced from the [National Park Service](https://www.nps.gov/): site-level 2025 recreation visits, monthly 2025 visits, and annual recreation visits from 1979–2025. See [`../README.md`](datasets/README.md) for expected files and columns.

**Scope:** This is descriptive analysis, not an estimate of unique visitors or a causal explanation for changes.

## 📂 2. Load the Data

Load the datasets from the `datasets/` folder. The CSVs must be added to the repository before running the notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

DATA_DIR = Path('datasets')
site_visits = pd.read_csv(DATA_DIR / 'dc_site_visits_2025.csv')
monthly_visits = pd.read_csv(DATA_DIR / 'dc_monthly_visits_2025.csv')
historical_visits = pd.read_csv(DATA_DIR / 'dc_historical_visits_1979_2025.csv')

site_visits.head()

## 🔍 3. Explore & Prepare the Data

Before interpreting the data, check the available columns, data types, missing values, and duplicate records. Preserve the original data and make any transformations explicit.

In [ ]:
for name, df in [('Sites', site_visits), ('Monthly', monthly_visits), ('Historical', historical_visits)]:
    print(f'\n{name}: {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(df.dtypes)
    print('Missing values:', df.isna().sum().to_dict())
    print('Duplicate rows:', df.duplicated().sum())

The monthly file may use month names or month numbers. The next cell puts months in calendar order for an accurate trend chart.

In [ ]:
month_names = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
monthly_visits = monthly_visits.copy()
if pd.api.types.is_numeric_dtype(monthly_visits['Month']):
    monthly_visits['MonthLabel'] = monthly_visits['Month'].map(dict(enumerate(month_names, 1)))
else:
    monthly_visits['MonthLabel'] = monthly_visits['Month'].astype(str).str.strip()
monthly_visits['MonthLabel'] = pd.Categorical(monthly_visits['MonthLabel'], categories=month_names, ordered=True)
monthly_visits = monthly_visits.sort_values('MonthLabel')
monthly_visits[['Month', 'MonthLabel', 'RecreationVisits']].head()

## 📊 4. Exploratory Data Analysis (EDA)

### How many sites and recreation visits are represented?

Start with two measures that establish the scale of visitation in the dataset.

In [ ]:
site_count = site_visits['ParkName'].nunique()
total_visits = site_visits['RecreationVisits'].sum()
print(f'NPS sites: {site_count:,}')
print(f'2025 recreation visits across sites: {total_visits:,.0f}')

### Which sites attract the most recreation visits?

Rank sites by total 2025 recreation visits. A horizontal bar chart makes long site names easier to read.

In [ ]:
top_sites = site_visits.sort_values('RecreationVisits', ascending=False).head(12)
fig = px.bar(top_sites.sort_values('RecreationVisits'), x='RecreationVisits', y='ParkName',
             orientation='h', title='Most Visited NPS Sites in the D.C. Area (2025)',
             labels={'RecreationVisits':'Recreation visits', 'ParkName':'NPS site'})
fig.update_traces(marker_color='#315d43')
fig.update_layout(yaxis_title='', xaxis_tickformat=',', template='plotly_white', height=550)
fig.show()

**Interpretation.** Visitation is not evenly distributed across sites. In the instructor analysis, the Lincoln Memorial leads, followed by George Washington Memorial Parkway. This concentration suggests that outreach and visitor-management needs may differ substantially by site.

### When is visitation highest?

Compare recreation visits by month to identify seasonal demand.

In [ ]:
fig = px.line(monthly_visits, x='MonthLabel', y='RecreationVisits', markers=True,
              title='Monthly Recreation Visits in the D.C. Area (2025)',
              labels={'MonthLabel':'Month', 'RecreationVisits':'Recreation visits'})
fig.update_traces(line_color='#315d43')
fig.update_layout(template='plotly_white', yaxis_tickformat=',')
fig.show()

**Interpretation.** Spring appears to be the busiest season, with a peak in April and elevated visits through May. This pattern could help guide the timing of visitor communications and seasonal staffing discussions.

### How has recreation visitation changed over time?

A historical view adds context that a single year cannot provide.

In [ ]:
fig = px.line(historical_visits.sort_values('Year'), x='Year', y='RecreationVisits',
              title='Annual Recreation Visits in the D.C. Area (1979–2025)',
              labels={'RecreationVisits':'Recreation visits'})
fig.update_traces(line_color='#315d43')
fig.update_layout(template='plotly_white', yaxis_tickformat=',')
fig.show()

**Interpretation.** The instructor analysis indicates a generally rising long-term pattern, a high point around 2016, and a sharp decline in 2020. The 2020 change is consistent with pandemic-era disruption, although this dataset alone cannot establish its cause.

## 💡 5. Key Findings & Insights

**1. Visits are concentrated at a relatively small number of sites.** The Lincoln Memorial and George Washington Memorial Parkway stand out in the 2025 site comparison. Site-specific planning may be more useful than treating every location the same.

**2. Spring is a peak visitation period.** April is the highest month in the instructor analysis, with May also elevated. Seasonal outreach and visitor information could be prioritized ahead of spring.

**3. Long-term context matters.** Annual recreation visits have changed substantially since 1979, including an unusual drop in 2020. A single-year snapshot cannot explain these shifts.

### Recommendation
Consider a **spring-focused visitor information campaign** for high-traffic sites, with clear guidance on visiting hours, transportation, and alternatives to crowded destinations. This is a proposal based on descriptive patterns, not a tested intervention.

### Limitations and next steps
- Recreation visits are **not necessarily unique people**; repeat visits can be counted.
- Site totals and monthly regional totals may use different reporting scopes; do not assume they reconcile without checking definitions.
- This analysis does not include weather, special events, transit conditions, or visitor demographics.
- Before using the findings operationally, verify the underlying CSVs and NPS reporting definitions, then examine whether patterns hold across multiple years.

**Data source:** [National Park Service](https://www.nps.gov/) · **Analysis:** Python, pandas, Plotly Express